# Lecture 01: Apache Spark Core Basics & RDDs
This notebook contains 20 distinct, sequential code cells showcasing the core configurations and RDD APIs of Apache Spark.

### 1. Initialize SparkSession
We configure the Spark Session with custom master threads and application metadata.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Lecture01_CoreBasics") \
    .master("local[4]") \
    .getOrCreate()
sc = spark.sparkContext
print("Spark Session Created!")

Spark Session Created!


### 2. Configure SparkContext Log Level
We reduce logging output to keep the console clean.

In [2]:
sc.setLogLevel("WARN")
print("Log level set to WARN successfully.")

Log level set to WARN successfully.


### 3. Check Spark UI Web URL
We retrieve the active link to monitor jobs and execution stages.

In [3]:
print("Spark Web UI Port:", sc.uiWebUrl)

Spark Web UI Port: http://spark-notebook:4041


### 4. Inspect Active JVM Configurations
Reading default configuration keys programmatically.

In [4]:
print("Serializer Config:", spark.conf.get("spark.serializer", "default_java"))

Serializer Config: org.apache.spark.serializer.KryoSerializer


### 5. Create RDD from Python List
Distributing local memory collections into an RDD.

In [5]:
list_rdd = sc.parallelize([10, 20, 30, 40, 50], numSlices=3)
print("Elements gathered:", list_rdd.collect())

Elements gathered: [10, 20, 30, 40, 50]


### 6. Inspect Partition Count
Getting the total number of partitions assigned to this RDD.

In [6]:
print("Number of Partitions:", list_rdd.getNumPartitions())

Number of Partitions: 3


### 7. Inspect Elements Layout per Partition (glom)
Gathering items in each partition block to see the distribution.

In [7]:
print("Glommed partitions data:", list_rdd.glom().collect())

Glommed partitions data: [[10], [20, 30], [40, 50]]


### 8. Map Transformation
Transforming elements row-by-row by squaring each number.

In [8]:
squared_rdd = list_rdd.map(lambda x: x ** 2)
print("Squared elements:", squared_rdd.collect())

Squared elements: [100, 400, 900, 1600, 2500]


### 9. Filter Transformation
Keeping only elements that are greater than or equal to 30.

In [ ]:
filtered_rdd = list_rdd.filter(lambda x: x >= 30)
print("Filtered elements:", filtered_rdd.collect())

### 10. FlatMap Tokenizer
Splitting sentences into individual words in a single flat sequence.

In [ ]:
sentences_rdd = sc.parallelize(["Hello distributed world", "Apache Spark is fast"])
words_rdd = sentences_rdd.flatMap(lambda sentence: sentence.split(" "))
print("Individual words:", words_rdd.collect())

### 11. mapPartitions Optimization
Processing elements batch-wise per partition to minimize expensive driver setup loops.

In [ ]:
def batch_db_insert(iterator):
    # Simulate initializing a connection once per partition
    connection = "MySQL_Active_Session"
    return [(x, connection) for x in iterator]
db_rdd = list_rdd.mapPartitions(batch_db_insert)
print("Processed partitions connection mappings:", db_rdd.collect())

### 12. mapPartitionsWithIndex
Accessing partition indices alongside the data iterator during processing.

In [ ]:
def show_indices(index, iterator):
    return [f"Partition {index} holds {val}" for val in iterator]
indexed_rdd = list_rdd.mapPartitionsWithIndex(show_indices)
print("Data placement details:", indexed_rdd.collect())

### 13. Action: reduce
Aggregating all elements together using a cumulative sum function.

In [ ]:
total_sum = list_rdd.reduce(lambda a, b: a + b)
print("Sum of elements:", total_sum)

### 14. Action: aggregate
Performing custom aggregate calculation returning sum and count in a single pass.

In [ ]:
seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))
sum_count = list_rdd.aggregate((0, 0), seqOp, combOp)
print("Aggregate result (Sum, Count):", sum_count)

### 15. MapReduce Simulation using reduceByKey
Performing word frequency aggregation.

In [ ]:
word_pairs = words_rdd.map(lambda w: (w.lower(), 1))
word_counts = word_pairs.reduceByKey(lambda a, b: a + b)
print("Word frequencies:", word_counts.collect())

### 16. GroupByKey (Wide dependency boundary)
Grouping values under each key, triggering full shuffle boundaries.

In [ ]:
grouped = word_pairs.groupByKey().mapValues(list)
print("Grouped keys:", grouped.collect())

### 17. SortByKey
Sorting the key-value pairs alphabetically by key.

In [ ]:
sorted_counts = word_counts.sortByKey(ascending=True)
print("Sorted frequencies:", sorted_counts.collect())

### 18. Cache and Uncache RDDs
Caching intermediate RDD to avoid recalculating the DAG.

In [ ]:
sorted_counts.cache()
print("Is RDD cached:", sorted_counts.is_cached)
sorted_counts.unpersist()
print("RDD uncached successfully.")

### 19. Create RDD from Local Text File
Reading raw text files directly.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
with open("data/mock_logs.txt", "w") as f:
    f.write("INFO: Job started\nERROR: Out of Memory\nINFO: Process complete")
logs_rdd = sc.textFile("data/mock_logs.txt")
print("Logs lines count:", logs_rdd.count())

### 20. Save RDD to a text directory
Writing out output files partitioned in local filesystem.

In [ ]:
import shutil
if os.path.exists("data/logs_output"):
    shutil.rmtree("data/logs_output")
logs_rdd.saveAsTextFile("data/logs_output")
print("Files inside logs_output directory:", os.listdir("data/logs_output"))

### 21. Clean up resources
Stopping Spark Session.

In [ ]:
spark.stop()